# Phase 2.4 : LSTM 24 h — **DÉPRÉCIÉ**

> **Ne pas exécuter.** Voir `TRAINING_PLAN.md` :
> - **Notebook 06** : modèles horaires **2 h** + **4 h** (`horizons_hourly`)
> - **Futur notebook 07** : LSTM **15 min**, lookback 12 h, horizon **1–2 h** (agrégats DB requis)

---

> Ancien texte : même pipeline que **06**, cible **24 pas** (= 24 h).

## Pourquoi un second modèle ?
- **4 h** : alerte opérationnelle rapide (ventilation, ajustement process).
- **24 h** : tendance journalière, planification, rapports.

## Différence technique
| | Notebook 06 | Notebook 07 |
|---|-------------|-------------|
| Horizon | 4 | 24 |
| Sortie Dense | `4 × 8 = 32` valeurs | `24 × 8 = 192` valeurs |
| Difficulté | Souvent plus facile | Erreur cumulée sur plus de pas |

## Attention débutant
Un horizon long = le modèle doit extrapoler plus loin → **RMSE souvent plus élevé** que le 4 h. Ce n'est pas un « échec », c'est normal.

## Sorties
- `models/model_lstm_24h.h5`
- `models/lstm_24h_metrics.json`

### Rappel

Si le notebook **05** ou **06** n'a pas été exécuté, le chargement du `.pkl` ou la comparaison 4h vs 24h échouera.

Exécutez les notebooks **dans l'ordre** : 05 → 06 → 07.

In [ ]:
# --- Setup identique au notebook 06, horizon=24 ---
import json
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow import keras
from tensorflow.keras import callbacks, layers

np.random.seed(42)
tf.random.set_seed(42)
sns.set_style("whitegrid")

PROJECT_ROOT = Path("../").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
from config import LSTM_CONFIG, N_FEATURES

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

HORIZON = LSTM_CONFIG["horizon_long"]
tensors_file = DATA_DIR / "lstm_train_val_test.pkl"
model_file = MODELS_DIR / "model_lstm_24h.h5"
metrics_file = MODELS_DIR / "lstm_24h_metrics.json"
history_file = MODELS_DIR / "lstm_24h_history.json"

print(f"TensorFlow {tf.__version__} | Horizon={HORIZON}h")

In [ ]:
# --- Charger tenseurs horizon 24 ---
with open(tensors_file, "rb") as f:
    lstm_data = pickle.load(f)

combined = lstm_data["combined_tensors"][HORIZON]
X_train, y_train = combined["train"]
X_val, y_val = combined["val"]
X_test, y_test = combined["test"]

for name, arr in [("X_train", X_train), ("y_train", y_train)]:
    if np.isnan(arr).any():
        raise ValueError(f"{name} contient des NaN — relancer notebook 05")

print(f"X_train {X_train.shape} → y_train {y_train.shape}")

### Tenseurs horizon 24

Clé `combined_tensors[24]` dans le pickle.

`y_train` a la forme `(N, 24, 8)` : pour chaque fenêtre, **24 heures futures** × 8 polluants.

In [ ]:
# --- Réseau : sortie (24, 8) au lieu de (4, 8) ---
dropout = LSTM_CONFIG["dropout_rate"]
lr = LSTM_CONFIG["learning_rate"]

model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1], X_train.shape[2])),
    layers.LSTM(LSTM_CONFIG["units_layer1"], return_sequences=True, activation="relu"),
    layers.Dropout(dropout),
    layers.LSTM(LSTM_CONFIG["units_layer2"], activation="relu"),
    layers.Dropout(dropout),
    layers.Dense(LSTM_CONFIG["dense_units"], activation="relu"),
    layers.Dense(HORIZON * N_FEATURES),
    layers.Reshape((HORIZON, N_FEATURES)),
])

model.compile(
    loss="mse",
    optimizer=keras.optimizers.Adam(learning_rate=lr),
    metrics=["mae"],
)
model.summary()

### Architecture identique au 4h

Seule la **dernière couche** change de taille (`24 * N_FEATURES` neurones).

Même hyperparamètres (`units`, `dropout`, `learning_rate`) depuis `config.py` — cohérence entre les deux modèles.

In [ ]:
# --- Entraînement (souvent plus long que 4h) ---
early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=LSTM_CONFIG["early_stopping_patience"],
    restore_best_weights=True,
    verbose=1,
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=LSTM_CONFIG["epochs"],
    batch_size=LSTM_CONFIG["batch_size"],
    callbacks=[early_stop],
    verbose=1,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history["loss"], label="Train")
axes[0].plot(history.history["val_loss"], label="Val")
axes[0].set_title("LSTM 24h — Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[1].plot(history.history["mae"], label="Train")
axes[1].plot(history.history["val_mae"], label="Val")
axes[1].set_title("LSTM 24h — MAE")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(MODELS_DIR / "lstm_24h_training_curves.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
# --- Erreurs pour histogramme et Q-Q plot ---
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test.reshape(-1), y_pred.reshape(-1))
rmse = float(np.sqrt(mse))
mae = float(mean_absolute_error(y_test.reshape(-1), y_pred.reshape(-1)))
r2 = float(r2_score(y_test.reshape(-1), y_pred.reshape(-1)))
mape = float(np.mean(np.abs((y_test - y_pred) / np.clip(np.abs(y_test), 1e-6, None))) * 100)
errors = np.abs(y_test - y_pred).flatten()

print(f"RMSE={rmse:.4f} | MAE={mae:.4f} | MAPE={mape:.2f}% | R²={r2:.4f}")

### Métriques test 24h

**MAPE** peut exploser si une valeur réelle est proche de 0 (division) — lire aussi **MAE** et **RMSE**.

Le notebook trace aussi la **distribution des erreurs** : utile pour voir si les grosses erreurs sont rares ou fréquentes.

In [ ]:
pollutants = lstm_data["pollutant_names"]
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
sample_idx = 0
hours = np.arange(1, HORIZON + 1)
for ax, p_idx in zip(axes.flatten(), range(min(4, N_FEATURES))):
    ax.plot(hours, y_test[sample_idx, :, p_idx], "o-", label="Réel", markersize=3)
    ax.plot(hours, y_pred[sample_idx, :, p_idx], "s--", label="Prédit", markersize=3, alpha=0.8)
    ax.set_title(pollutants[p_idx])
    ax.set_xlabel("Heure (+h)")
    ax.legend()
    ax.grid(alpha=0.3)
plt.suptitle("LSTM 24h — exemple")
plt.tight_layout()
plt.savefig(MODELS_DIR / "lstm_24h_predictions.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(errors, bins=50, color="steelblue", alpha=0.7, edgecolor="black")
axes[0].set_title("Distribution erreurs absolues")
stats.probplot(errors, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q plot")
plt.tight_layout()
plt.savefig(MODELS_DIR / "lstm_24h_error_distribution.png", dpi=100, bbox_inches="tight")
plt.show()

### Distribution des erreurs + Q-Q plot

- **Histogramme** : erreurs absolues concentrées près de 0 ?
- **Q-Q plot** : compare la distribution des erreurs à une loi normale (diagnostic avancé, pas bloquant).

In [ ]:
model.save(model_file)

metrics = {
    "horizon_steps": HORIZON,
    "horizon_hours": HORIZON,
    "lookback_hours": lstm_data["lookback"],
    "timestep_minutes": lstm_data["timestep_minutes"],
    "rmse": rmse,
    "mae": mae,
    "mape": mape,
    "r2_score": r2,
    "test_samples": int(len(y_test)),
    "epochs_trained": len(history.history["loss"]),
    "error_mean": float(np.mean(errors)),
    "error_median": float(np.median(errors)),
}

with open(metrics_file, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

with open(history_file, "w", encoding="utf-8") as f:
    json.dump(history.history, f, indent=2)

print(f"Modèle: {model_file}")

In [ ]:
# --- Comparaison optionnelle avec le modèle 4h ---
metrics_4h_file = MODELS_DIR / "lstm_4h_metrics.json"
if metrics_4h_file.exists():
    with open(metrics_4h_file, encoding="utf-8") as f:
        m4 = json.load(f)
    comparison = pd.DataFrame([
        {"Model": "LSTM 4h", "RMSE": m4["rmse"], "MAE": m4["mae"], "MAPE (%)": m4["mape"], "R²": m4["r2_score"]},
        {"Model": "LSTM 24h", "RMSE": metrics["rmse"], "MAE": metrics["mae"], "MAPE (%)": metrics["mape"], "R²": metrics["r2_score"]},
    ])
    print(comparison.to_string(index=False))
else:
    print("Exécuter d'abord le notebook 06 pour comparer 4h vs 24h.")

### Comparaison 4h vs 24h

Tableau récapitulatif si `lstm_4h_metrics.json` existe.

En général : **4h** = métriques meilleures ; **24h** = vision plus stratégique.

## Fin du bloc LSTM (notebooks 05–07)

**Vous savez maintenant :**
1. Préparer des fenêtres temporelles (05).
2. Entraîner un LSTM multi-sorties (06–07).
3. Lire loss, MAE, R² et courbes de prédiction.

**Suite du projet :** notebook **08** (inférence + latence) et **09** (validation bout en bout).

**Documentation complémentaire :** `docs/DISCUSSION_MODELES_IA_RECOMMANDATIONS.md`